# Amazon ESCI Data EDA
This notebook performs EDA on the Amazon ESCI dataset.
It covers loading the data, inspecting E/S/C/I distributions, candidates per query, and filtering by locale.

In [2]:
import pandas as pd
import numpy as np

# Load datasets
products_path = '../data/raw/shopping_queries_dataset_products.parquet'
examples_path = '../data/raw/shopping_queries_dataset_examples.parquet'
sources_path = '../data/raw/shopping_queries_dataset_sources.csv'

print("Loading datasets...")
df_products = pd.read_parquet(products_path)
df_examples = pd.read_parquet(examples_path)
df_sources = pd.read_csv(sources_path)

print("Products shape:", df_products.shape)
print("Examples shape:", df_examples.shape)
print("Sources shape:", df_sources.shape)

Loading datasets...
Products shape: (1814924, 7)
Examples shape: (2621288, 9)
Sources shape: (130652, 2)


In [12]:
df_products.head()

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,B079VKKJN7,"11 Degrees de los Hombres Playera con Logo, Ne...",Esta playera con el logo de la marca Carrier d...,11 Degrees Negro Playera con logo\nA estrenar ...,11 Degrees,Negro,es
1,B079Y9VRKS,Camiseta Eleven Degrees Core TS White (M),NaN,NaN,11 Degrees,Blanco,es
2,B07DP4LM9H,11 Degrees de los Hombres Core Pull Over Hoodi...,La sudadera con capucha Core Pull Over de 11 G...,11 Degrees Azul Core Pull Over Hoodie\nA estre...,11 Degrees,Azul,es
3,B07G37B9HP,11 Degrees Poli Panel Track Pant XL Black,NaN,NaN,11 Degrees,NaN,es
4,B07LCTGDHY,11 Degrees Gorra Trucker Negro OSFA (Talla úni...,NaN,NaN,11 Degrees,Negro (,es


In [13]:
df_examples.head()

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train


In [14]:
df_sources.head()

,query_id,source
0,0,other
1,1,negations
2,2,negations
3,3,negations
4,4,behavioral


In [3]:
# 1. E/S/C/I distribution
print("ESCI Label Distribution (%):")
print(df_examples['esci_label'].value_counts(normalize=True) * 100)

ESCI Label Distribution (%):
esci_label
E    65.164835
S    21.909573
I    10.039530
C     2.886062
Name: proportion, dtype: float64


In [4]:
# 2. Average candidates per query
avg_candidates = df_examples.groupby('query_id').size().mean()
print(f"Average candidates per query: {avg_candidates:.2f}")

Average candidates per query: 20.06


In [5]:
# 3. Locale distribution & Filtering
print("Locale Distribution before filtering:")
print(df_examples['product_locale'].value_counts())

print("\nFiltering to 'us' locale...")
df_examples_us = df_examples[df_examples['product_locale'] == 'us'].copy()
print("Examples shape after filtering:", df_examples_us.shape)

print("\nChecking locale distribution in products:")
print(df_products['product_locale'].value_counts())

df_products_us = df_products[df_products['product_locale'] == 'us'].copy()
print("Products shape after filtering:", df_products_us.shape)

Locale Distribution before filtering:
product_locale
us    1818825
jp     446053
es     356410
Name: count, dtype: int64

Filtering to 'us' locale...
Examples shape after filtering: (1818825, 9)

Checking locale distribution in products:
product_locale
us    1215854
jp     339059
es     260011
Name: count, dtype: int64
Products shape after filtering: (1215854, 7)


In [6]:
# 4. Small sample test vs Full test
def process_queries(df_ex, df_prod, num_queries):
    # Get a sample of unique queries
    sample_qids = df_ex['query_id'].unique()[:num_queries]
    df_sample_ex = df_ex[df_ex['query_id'].isin(sample_qids)]
    
    # Get corresponding products
    sample_asins = df_sample_ex['product_id'].unique()
    df_sample_prod = df_prod[df_prod['product_id'].isin(sample_asins)]
    
    print(f"\n--- Processing {num_queries} queries ---")
    print("Examples shape:", df_sample_ex.shape)
    print("Products shape:", df_sample_prod.shape)
    print("\nMissing values in examples:\n", df_sample_ex.isnull().sum())
    print("\nMissing values in products:\n", df_sample_prod.isnull().sum())
    return df_sample_ex, df_sample_prod

print("Testing on 200 queries...")
df_200_ex, df_200_prod = process_queries(df_examples_us, df_products_us, 200)

print("\nTesting on 4000 queries...")
df_4000_ex, df_4000_prod = process_queries(df_examples_us, df_products_us, 4000)

Testing on 200 queries...

--- Processing 200 queries ---
Examples shape: (5977, 9)
Products shape: (5398, 7)

Missing values in examples:
 example_id        0
query             0
query_id          0
product_id        0
product_locale    0
esci_label        0
small_version     0
large_version     0
split             0
dtype: int64

Missing values in products:
 product_id                 0
product_title              0
product_description     2630
product_bullet_point     454
product_brand            247
product_color           1499
product_locale             0
dtype: int64

Testing on 4000 queries...

--- Processing 4000 queries ---
Examples shape: (89723, 9)
Products shape: (75120, 7)

Missing values in examples:
 example_id        0
query             0
query_id          0
product_id        0
product_locale    0
esci_label        0
small_version     0
large_version     0
split             0
dtype: int64

Missing values in products:
 product_id                  0
product_title          

## Additional check

In [8]:
ex = pd.read_parquet(examples_path)
us = ex[ex.small_version == 1] if "small_version" in ex else ex
us = us[us.product_locale == "us"]
has_c = us.groupby("query_id").esci_label.apply(lambda s: (s == "C").any())
print(f"queries with at least one C: {has_c.mean():.1%}  (need >= 20%)")

queries with at least one C: 25.8%  (need >= 20%)


In [9]:
ex = pd.read_parquet(examples_path)
us = ex[ex.product_locale == "us"]
g = us.groupby("query_id").esci_label
print(f"queries with >=1 C: {g.apply(lambda s: (s=='C').any()).mean():.1%}")
print(f"median candidates/query: {g.size().median():.0f}")

queries with >=1 C: 13.8%
median candidates/query: 16


In [10]:
e_frac = us.groupby("query_id").esci_label.apply(lambda s: (s=='E').mean())
threshold = e_frac.quantile(0.40)          # 下 60% 即为 hard
print(f"difficulty threshold: E-fraction <= {threshold:.3f}")

difficulty threshold: E-fraction <= 0.688


In [11]:
ex = pd.read_parquet(examples_path)
df = ex[(ex.small_version == 1) & (ex.product_locale == "us")]

g = df.groupby("query_id")
e_frac, cand = g.esci_label.apply(lambda s: (s=="E").mean()), g.size()

print("queries           :", len(cand))
print("has C             : {:.1%}".format(g.esci_label.apply(lambda s:(s=="C").any()).mean()))
print("difficulty thresh : E-frac <= {:.3f}".format(e_frac.quantile(0.40)))
print("candidates median : {}   >=20: {:.1%}".format(cand.median(), (cand>=20).mean()))

# 直接测去重后的商品数，按 2000 条查询采样再线性外推
samp = np.random.default_rng(42).choice(cand.index, 2000, replace=False)
n2k = df[df.query_id.isin(samp)].product_id.nunique()
print(f"unique products @2000 queries: {n2k:,}  → @4000 est. {int(n2k*1.9):,}")

queries           : 29844
has C             : 25.8%
difficulty thresh : E-frac <= 0.312
candidates median : 16.0   >=20: 19.5%
unique products @2000 queries: 38,571  → @4000 est. 73,284
